In [4]:
import ee
import geedim
import os
import pandas as pd
import numpy as np
from glob import glob
from pathlib import Path

# Image Processing and Machine Learning
from skimage import io
from skimage.segmentation import morphological_chan_vese, checkerboard_level_set
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib_scalebar.scalebar import ScaleBar
from mpl_toolkits.axes_grid1 import make_axes_locatable
import cmcrameri.cm as cmc

ee.Initialize(opt_url='https://earthengine-highvolume.googleapis.com')

In [9]:
base_path = 'data'
os.makedirs(base_path, exist_ok=True)

# 1. Define Study Areas and Download Satellite Image

In [5]:
study_areas = { 
    # "name": ([lat, lon], date_idx, buffer_in_km)
    "sundarbans_bangladesh" : ([ 21.6486,   88.9905], 0, 5), 
    "shanghai_port"         : ([ 30.6173,  122.0843], 0, 5),
    "wadden_sea"            : ([ 53.5116,    6.4813], 0, 5),
    "great_barrier_reef"    : ([-19.7347,  149.1586], 0, 5),
    "maranhenses"           : ([ -2.5040,  -42.9548], 0, 5),

    # * Feel free to add more study areas or using different date_idx and buffer_in_km

    # date_idx is for the date picking from the prop_table_split, default is 0
    # buffer_in_km is for the buffer size around the point, default is 5
}

In [ ]:
year = 2024
start_date = f"{year}-01-01"
end_date   = f"{year+1}-01-01"

coll = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
download = True # Toggle to False if data is already downloaded

for area_name, (region, date_idx, buffer) in study_areas.items():
    buffer_m = buffer * 1000 
    y_cen, x_cen = region
    geom = ee.Geometry.Point(x_cen, y_cen).buffer(buffer_m).bounds()
    
    # Filter collection based on region and cloudless portion using geedim
    single_filt_coll = coll.gd.filter(
        start_date, end_date, region=geom, cloudless_portion=95
    )
    
    # Parse the properties table to extract valid dates
    prop_table = single_filt_coll.gd.propertiesTable.split("\n")[2:]
    prop_table_split = [list(filter(None, s.split(" ")))[1] for s in prop_table]
    prop_table_split = list(set(prop_table_split))
    
    target_date = pd.to_datetime(prop_table_split[date_idx])
    print(f"Target date for {area_name}: {target_date.strftime('%Y-%m-%d')}")

    # Narrow down the collection to the specific target date
    single_filt_coll = coll.gd.filter(
        (target_date - pd.Timedelta(days=1)).strftime("%Y-%m-%d"), 
        (target_date + pd.Timedelta(days=1)).strftime("%Y-%m-%d"), 
        region=geom
    )

    comp_im = single_filt_coll.gd.composite().toInt16()
    tif_save_path = f"{base_path}/{area_name}_{target_date.strftime('%Y-%m-%d')}.tif"
    
    # Download the selected bands
    if download and not os.path.exists(tif_save_path):
        prep_im = comp_im.gd.prepareForExport(
            crs='EPSG:3395', 
            region=geom, 
            scale=10, 
            dtype='uint16',
            bands=["B1", "B2", "B3", "B4", "B8", "B11", "B12"],
        )
        prep_im.gd.toGeoTIFF(tif_save_path)
        print(f"Successfully downloaded: {tif_save_path}")

# Shoreline extraction & Pixel-wise Confidence

In [10]:
def MorphACWE(image, checkerboard_level=5, MACWE_iteration=2, MACWE_smooth=1):
    """
    Extracts binary shoreline contours using Morphological Chan-Vese.
    """
    init_ls = checkerboard_level_set(image.shape, checkerboard_level)
    
    ls = morphological_chan_vese(
        image, 
        num_iter=MACWE_iteration, 
        init_level_set=init_ls, 
        smoothing=MACWE_smooth
    )
    return ls

In [11]:
def process_and_plot(tif_path):
    """Applies PCA and MACWE, then generates a perfectly aligned tutorial visualization."""
    file_name = Path(tif_path).stem
    print(f"\nProcessing Visualization for: {file_name}")

    # 1. Load the multi-band image
    layered_img = io.imread(tif_path).astype(np.float32)

    nbands, nx, ny = layered_img.shape  
    layered_img_shaped = layered_img.reshape((nbands, nx * ny))

    # 2. Rescale Data and Apply PCA
    scaler = StandardScaler()
    img_rescaled = scaler.fit_transform(layered_img_shaped) 

    # Fit PCA keeping 90% of the variance
    pca = PCA(n_components=0.9).fit(img_rescaled.T)
    
    # 3. Process First Principal Component (PC1)
    pc1 = pca.components_[0]
    pc1_proj = (pc1 @ img_rescaled).reshape(nx, ny)

    # Polarity correction based on the NIR band (band index 0 in rescaled data)
    NIR_norm = img_rescaled[0].reshape(nx, ny)
    if NIR_norm[pc1_proj > 0].mean() < NIR_norm[pc1_proj < 0].mean():
        pc1_proj = -pc1_proj

    # 4. Extract Shoreline with MACWE
    ls_PCA = MorphACWE(pc1_proj)
    
    # Polarity correction for the segmented output using the NIR band
    if NIR_norm[ls_PCA == 0].mean() < NIR_norm[ls_PCA != 0].mean():
        ls_PCA = -ls_PCA

    # 5. Generate the Tutorial Plot with Uniform Sizing
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # Plot A: RGB Image
    jpg_img = layered_img[[3, 2, 1], :, :].transpose(1, 2, 0)
    jpg_img = np.clip(jpg_img, 0, 2000) / 2000 # Clip at 2000 for better visualization
    axes[0].imshow(jpg_img)
    axes[0].set_title("RGB Composite")
    axes[0].axis('off')
    
    # Add a dummy invisible colorbar axis to Plot A to maintain size parity
    div0 = make_axes_locatable(axes[0])
    cax0 = div0.append_axes("right", size="5%", pad=0.1)
    cax0.axis('off')

    # Plot B: PCA Confidence / Uncertainty Map
    uncertainty_map = np.sqrt(pc1_proj**2)
    norm_uncertainty = uncertainty_map / np.max(uncertainty_map)
    im_conf = axes[1].imshow(norm_uncertainty, vmin=0, vmax=1, cmap=cmc.batlowW)
    axes[1].set_title(f"Norm. Confidence Map\n(Raw Max: {uncertainty_map.max():.2f})")
    axes[1].axis('off')
    
    # Add the actual colorbar to Plot B
    div1 = make_axes_locatable(axes[1])
    cax1 = div1.append_axes("right", size="5%", pad=0.1)
    plt.colorbar(im_conf, cax=cax1)

    # Plot C: Binary Shoreline Map
    axes[2].imshow(ls_PCA, cmap=cmc.batlow)
    axes[2].set_title("Land-Water Boundary")
    axes[2].axis('off')
    
    # Add a dummy invisible colorbar axis to Plot C to maintain size parity
    div2 = make_axes_locatable(axes[2])
    cax2 = div2.append_axes("right", size="5%", pad=0.1)
    cax2.axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
# Execute the plotting pipeline for all files in the data directory
tif_paths = sorted(glob(f"{base_path}/*.tif"))
for tif_path in tif_paths:
    process_and_plot(tif_path)